# 🤖 LIAR Dataset — Model Training Pipeline
## Fake News Detection Using NLP Classifiers — Project #13

---

### 📌 What This Notebook Does

This notebook trains **three separate models** on the LIAR dataset features prepared in the previous notebooks:

| # | Model | Features Used | Expected Accuracy |
|---|-------|--------------|------------------|
| 1 | 🔵 Logistic Regression | TF-IDF + Hand-crafted (scaled) | ~63–66% |
| 2 | 🟠 Gradient Boosting | Hand-crafted features only | ~65–68% |
| 3 | 🟣 BERT (DistilBERT) | Raw statement text (fine-tuned) | ~68–72% |

### 📋 Prerequisites
Before running this notebook, make sure you have run:
1. `LIAR_Preprocessing.ipynb` → produces `/content/cleaned_datasets/`
2. `LIAR_Feature_Extraction.ipynb` → produces `/content/feature_store/`

### 🗂️ Notebook Structure
- **Part 1** — Setup & data loading
- **Part 2** — Logistic Regression (trained alone, fully evaluated)
- **Part 3** — Gradient Boosting (trained alone, fully evaluated)
- **Part 4** — BERT / DistilBERT (trained alone, fully evaluated)
- **Part 5** — Cross-model comparison

---
> 💡 **Each model section is self-contained.** You can run them independently.


---
## ⚙️ Part 1 — Environment Setup

### Step 1.1 — Install Required Libraries

We install everything needed across all three models in one go.
- `scikit-learn` → Logistic Regression + Gradient Boosting + metrics
- `transformers` + `torch` → BERT / DistilBERT fine-tuning
- `imbalanced-learn` → already used in preprocessing, kept for reference
- `seaborn` + `matplotlib` → visualisations


In [ ]:
# ── Install all required packages ──────────────────────────────────────────
# This may take 1-2 minutes on Colab the first time

!pip install -q scikit-learn imbalanced-learn seaborn matplotlib joblib
!pip install -q transformers datasets torch torchvision torchaudio
!pip install -q accelerate

print("✅ All packages installed.")


### Step 1.2 — Import Libraries

We import everything at the top so there are no surprise errors halfway through training.


In [ ]:
# ── Standard libraries ──────────────────────────────────────────────────────
import os
import warnings
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import scipy.sparse as sp
from collections import Counter

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# ── Scikit-learn — Logistic Regression ──────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score, precision_score, recall_score
)

# ── Scikit-learn — Gradient Boosting ────────────────────────────────────────
from sklearn.ensemble import GradientBoostingClassifier

# ── HuggingFace Transformers — BERT ─────────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW

# ── GPU availability check ───────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Libraries imported.")
print(f"🖥️  Device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   ⚠️  No GPU detected — BERT training will be slow. Enable GPU in:")
    print("      Runtime → Change runtime type → Hardware accelerator → GPU")


### Step 1.3 — Mount Google Drive & Set Paths

We mount Google Drive so our trained models are saved permanently.  
Update `BASE_DIR` if you saved the feature store somewhere else.


In [ ]:
# ── Mount Google Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Set directory paths ───────────────────────────────────────────────────────
# Change these paths if you saved your files in a different location
FEATURE_DIR  = '/content/feature_store'      # Output from Feature Extraction notebook
CLEANED_DIR  = '/content/cleaned_datasets'   # Output from Preprocessing notebook
MODEL_DIR    = '/content/drive/MyDrive/LIAR_Models'  # Where trained models are saved

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(f'{MODEL_DIR}/logistic_regression', exist_ok=True)
os.makedirs(f'{MODEL_DIR}/gradient_boosting',   exist_ok=True)
os.makedirs(f'{MODEL_DIR}/bert',                exist_ok=True)

print(f"✅ Paths configured.")
print(f"   Feature store : {FEATURE_DIR}")
print(f"   Cleaned data  : {CLEANED_DIR}")
print(f"   Model output  : {MODEL_DIR}")


### Step 1.4 — Load Feature Matrices

We load the pre-built sparse feature matrices from the Feature Extraction notebook.
These contain:
- **TF-IDF features** (~11,000 sparse columns)
- **Hand-crafted features** (~80 dense columns), scaled with MinMaxScaler


In [ ]:
# ── Load sparse feature matrices (TF-IDF + hand-crafted) ─────────────────────
print("⏳ Loading feature matrices...")

X_train = sp.load_npz(f'{FEATURE_DIR}/X_train.npz')
X_valid = sp.load_npz(f'{FEATURE_DIR}/X_valid.npz')
X_test  = sp.load_npz(f'{FEATURE_DIR}/X_test.npz')

y_train = np.load(f'{FEATURE_DIR}/y_train.npy')
y_valid = np.load(f'{FEATURE_DIR}/y_valid.npy')
y_test  = np.load(f'{FEATURE_DIR}/y_test.npy')

print("✅ Feature matrices loaded.")
print(f"   X_train : {X_train.shape}  —  labels: {Counter(y_train)}")
print(f"   X_valid : {X_valid.shape}  —  labels: {Counter(y_valid)}")
print(f"   X_test  : {X_test.shape}  —  labels: {Counter(y_test)}")

# ── Load enriched CSV for BERT text + hand-crafted-only experiments ───────────
print("\n⏳ Loading enriched CSVs...")
train_df = pd.read_csv(f'{FEATURE_DIR}/train_features.csv')
valid_df = pd.read_csv(f'{FEATURE_DIR}/valid_features.csv')
test_df  = pd.read_csv(f'{FEATURE_DIR}/test_features.csv')

print("✅ CSVs loaded.")
print(f"   train_df : {train_df.shape}")
print(f"   Columns preview : {list(train_df.columns[:10])} ...")


### Step 1.5 — Helper: Evaluation Function

This function is shared by all three models.  
It prints a full classification report, confusion matrix, and ROC-AUC score.


In [ ]:
# ── Shared evaluation helper ──────────────────────────────────────────────────
# This function is used by ALL three models — we define it once here.

def evaluate_model(model_name, y_true, y_pred, y_prob=None, split='Test'):
    """
    Print a full evaluation report for any binary classifier.
    
    Parameters
    ----------
    model_name : str   — display name for the model
    y_true     : array — ground truth labels (0/1)
    y_pred     : array — predicted labels (0/1)
    y_prob     : array — predicted probabilities for class 1 (optional, for ROC)
    split      : str   — 'Validation' or 'Test'
    """
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)

    print(f"\n{'='*55}")
    print(f"  📊 {model_name} — {split} Results")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    if y_prob is not None:
        auc = roc_auc_score(y_true, y_prob)
        print(f"  ROC-AUC   : {auc:.4f}")
    print(f"{'='*55}")
    print()
    print(classification_report(y_true, y_pred, target_names=['FAKE','REAL']))

    # ── Confusion matrix plot ─────────────────────────────────────────────────
    cm = confusion_matrix(y_true, y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Left: confusion matrix heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['FAKE','REAL'], yticklabels=['FAKE','REAL'],
                ax=axes[0], linewidths=0.5)
    axes[0].set_xlabel('Predicted Label')
    axes[0].set_ylabel('True Label')
    axes[0].set_title(f'{model_name}\nConfusion Matrix ({split})', fontweight='bold')

    # Right: ROC curve (if probabilities provided)
    if y_prob is not None:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc = roc_auc_score(y_true, y_prob)
        axes[1].plot(fpr, tpr, color='#1976d2', lw=2, label=f'ROC (AUC = {auc:.3f})')
        axes[1].plot([0,1],[0,1],'--', color='grey', lw=1, label='Random baseline')
        axes[1].fill_between(fpr, tpr, alpha=0.1, color='#1976d2')
        axes[1].set_xlabel('False Positive Rate')
        axes[1].set_ylabel('True Positive Rate')
        axes[1].set_title(f'{model_name}\nROC Curve ({split})', fontweight='bold')
        axes[1].legend(fontsize=10)
    else:
        axes[1].axis('off')

    plt.tight_layout()
    plt.show()

    return {'accuracy': acc, 'f1': f1, 'precision': prec, 'recall': rec}


print("✅ evaluate_model() helper defined — ready to use for all models.")


---
## 🔵 Part 2 — Logistic Regression

### What is Logistic Regression?
Logistic Regression is a **linear classifier** — it finds the best straight-line boundary  
that separates FAKE from REAL news in the feature space.

**Why use it for NLP?**
- Works extremely well with **sparse TF-IDF** vectors (tens of thousands of features)
- Fast to train — seconds on the full LIAR dataset
- Highly interpretable — we can see which words push the prediction toward FAKE or REAL
- Strong baseline that often beats more complex models on short text

**Feature input:** The full sparse matrix `X_train.npz` (TF-IDF + hand-crafted scaled features)

---


### Step 2.1 — Train Logistic Regression

We use `saga` solver which handles **L1/L2 regularisation** and large sparse matrices efficiently.

Key hyperparameters:
- `C=1.0` — regularisation strength (smaller = more regularisation, prevents overfitting)
- `class_weight='balanced'` — adjusts for label imbalance automatically
- `max_iter=1000` — enough iterations for saga to converge on this dataset


In [ ]:
# ── Train Logistic Regression ────────────────────────────────────────────────
# This should take about 10-30 seconds on Colab

print("🔵 Training Logistic Regression...")
print("   Input  : X_train.npz (TF-IDF + hand-crafted features)")
print(f"  Shape  : {X_train.shape}")
print()

start = time.time()

lr_model = LogisticRegression(
    C=1.0,                    # Regularisation — controls overfitting
    solver='saga',            # Best solver for large sparse matrices
    max_iter=1000,            # Max iterations for convergence
    class_weight='balanced',  # Handles FAKE/REAL class imbalance
    random_state=42,
    n_jobs=-1                 # Use all CPU cores
)

lr_model.fit(X_train, y_train)

elapsed = time.time() - start
print(f"✅ Logistic Regression trained in {elapsed:.1f} seconds.")
print(f"   Iterations used: {lr_model.n_iter_[0]}")


### Step 2.2 — Validate on Validation Set

We always evaluate on the **validation set first** — it's the "dev set" we tune on.  
The **test set** is only used once at the very end to get the final honest score.


In [ ]:
# ── Evaluate on validation set ───────────────────────────────────────────────
print("🔍 Evaluating on Validation Set...")

lr_val_pred = lr_model.predict(X_valid)
lr_val_prob = lr_model.predict_proba(X_valid)[:, 1]  # Probability of class 1 (REAL)

lr_val_scores = evaluate_model(
    model_name='Logistic Regression',
    y_true=y_valid,
    y_pred=lr_val_pred,
    y_prob=lr_val_prob,
    split='Validation'
)


### Step 2.3 — Final Test Set Evaluation

Now we run once on the held-out test set for the final honest result.


In [ ]:
# ── Evaluate on test set ─────────────────────────────────────────────────────
print("🏁 Evaluating on Test Set (final honest score)...")

lr_test_pred = lr_model.predict(X_test)
lr_test_prob = lr_model.predict_proba(X_test)[:, 1]

lr_test_scores = evaluate_model(
    model_name='Logistic Regression',
    y_true=y_test,
    y_pred=lr_test_pred,
    y_prob=lr_test_prob,
    split='Test'
)


### Step 2.4 — Interpret: Top Predictive Words

One of the best things about Logistic Regression is **interpretability**.  
The model learned a coefficient for each TF-IDF feature (word).  
- **Positive coefficients** → push prediction toward REAL  
- **Negative coefficients** → push prediction toward FAKE

We visualise the top 20 words for each direction.


In [ ]:
# ── Visualise top predictive words ────────────────────────────────────────────
# Load the TF-IDF vectoriser to get feature names
vec_unigram = joblib.load(f'{FEATURE_DIR}/tfidf_unigram.pkl')
vec_bigram  = joblib.load(f'{FEATURE_DIR}/tfidf_bigram.pkl')
vec_char    = joblib.load(f'{FEATURE_DIR}/tfidf_char.pkl')
scaler      = joblib.load(f'{FEATURE_DIR}/scaler.pkl')

# Reconstruct all feature names in the same order as X_train columns
feature_names = (
    [f'uni:{w}' for w in vec_unigram.get_feature_names_out()] +
    [f'bi:{w}'  for w in vec_bigram.get_feature_names_out()] +
    [f'char:{w}'for w in vec_char.get_feature_names_out()]
)

# Pad with 'hc:...' names for hand-crafted features at the end
n_hc = X_train.shape[1] - len(feature_names)
feature_names += [f'hc:{i}' for i in range(n_hc)]

coefs = lr_model.coef_[0]

# Top features for REAL (positive coef) and FAKE (negative coef)
# Only show unigrams for clarity (most meaningful)
uni_mask = [i for i, n in enumerate(feature_names) if n.startswith('uni:')]
uni_coefs = [(feature_names[i].replace('uni:',''), coefs[i]) for i in uni_mask]
uni_coefs.sort(key=lambda x: x[1])

top_fake = uni_coefs[:20]    # Most negative = most FAKE
top_real = uni_coefs[-20:]   # Most positive = most REAL

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# FAKE words
fake_words, fake_vals = zip(*top_fake)
axes[0].barh(fake_words, fake_vals, color='#e57373', edgecolor='white')
axes[0].set_title('🔴 Top 20 Words → Predicts FAKE\n(Most Negative Coefficients)', fontweight='bold')
axes[0].set_xlabel('Coefficient Value')
axes[0].axvline(0, color='black', lw=0.8)

# REAL words
real_words, real_vals = zip(*reversed(top_real))
axes[1].barh(real_words, real_vals, color='#81c784', edgecolor='white')
axes[1].set_title('🟢 Top 20 Words → Predicts REAL\n(Most Positive Coefficients)', fontweight='bold')
axes[1].set_xlabel('Coefficient Value')
axes[1].axvline(0, color='black', lw=0.8)

plt.suptitle('📝 Logistic Regression — Most Predictive Words', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Words with large positive coefficients strongly push toward REAL.")
print("   Words with large negative coefficients strongly push toward FAKE.")


### Step 2.5 — Save the Model


In [ ]:
# ── Save Logistic Regression model to Google Drive ────────────────────────────
lr_save_path = f'{MODEL_DIR}/logistic_regression/lr_model.pkl'
joblib.dump(lr_model, lr_save_path)

# Save scores summary
with open(f'{MODEL_DIR}/logistic_regression/scores.json', 'w') as f:
    json.dump({'validation': lr_val_scores, 'test': lr_test_scores}, f, indent=2)

print(f"✅ Logistic Regression model saved to: {lr_save_path}")
print(f"   Val  Accuracy : {lr_val_scores['accuracy']:.4f}")
print(f"   Test Accuracy : {lr_test_scores['accuracy']:.4f}")


---
## 🟠 Part 3 — Gradient Boosting

### What is Gradient Boosting?
Gradient Boosting builds an **ensemble of decision trees** one by one.  
Each new tree tries to fix the mistakes of the previous trees — this is called "boosting".

**Why use it here?**
- Excels on **tabular / hand-crafted features** (speaker credibility, readability, linguistics)
- Doesn't need sparse matrices — we use the dense hand-crafted features here
- Captures **non-linear relationships** between features that Logistic Regression misses
- Feature importance scores are built in

**Feature input:** Hand-crafted dense features only (not the full sparse matrix)

> ⚠️ We use sklearn's `GradientBoostingClassifier` here.  
> If you want faster training, you can swap in `lightgbm.LGBMClassifier` instead.

---


### Step 3.1 — Prepare Hand-Crafted Feature Matrix

Gradient Boosting works best with **dense** tabular features, not sparse TF-IDF.  
We extract the hand-crafted columns from our enriched CSV.


In [ ]:
# ── Select hand-crafted features for Gradient Boosting ────────────────────────
# These are the non-TF-IDF features: linguistic, readability, credit, categorical

# Define the same set used in the feature extraction notebook
HANDCRAFTED_COLS = (
    # From preprocessing
    ['word_count','char_count','avg_word_length','unique_word_ratio',
     'sentence_count','avg_words_per_sent','uppercase_ratio',
     'exclamation_count','question_count','punctuation_density',
     'has_quote','has_number','number_count','has_percent',
     'flesch_reading_ease','gunning_fog','smog_index','flesch_kincaid_grade',
     'credit_total','credit_fake_ratio','credit_true_ratio','credibility_score',
     'subject_count','party_known','state_known',
     'speaker_enc','party_enc','state_enc','job_title_enc'] +
    # Linguistic features (from feature extraction notebook)
    [c for c in train_df.columns if c.startswith('ling_')] +
    # Sentiment proxies
    [c for c in train_df.columns if c.startswith('sent_')] +
    # Credibility features
    [c for c in train_df.columns if c.startswith('cred_')] +
    # Frequency encodings
    [c for c in train_df.columns if c.endswith('_freq')] +
    # Raw credit counts
    ['barely_true_count','false_count','half_true_count',
     'mostly_true_count','pants_on_fire_count']
)

# Keep only those that actually exist in the CSV
HANDCRAFTED_COLS = list(dict.fromkeys(
    c for c in HANDCRAFTED_COLS if c in train_df.columns
))

print(f"✅ Using {len(HANDCRAFTED_COLS)} hand-crafted features for Gradient Boosting")
print(f"   Sample columns: {HANDCRAFTED_COLS[:8]}")

# Build dense feature matrices
X_gb_train = train_df[HANDCRAFTED_COLS].fillna(0).values
X_gb_valid = valid_df[HANDCRAFTED_COLS].fillna(0).values
X_gb_test  = test_df[HANDCRAFTED_COLS].fillna(0).values

print(f"\n   X_gb_train shape: {X_gb_train.shape}")
print(f"   X_gb_valid shape: {X_gb_valid.shape}")
print(f"   X_gb_test  shape: {X_gb_test.shape}")


### Step 3.2 — Train Gradient Boosting

Key hyperparameters:
- `n_estimators=300` — number of trees to build (more = better, but slower)
- `learning_rate=0.05` — how much each tree contributes (small = more conservative, less overfitting)
- `max_depth=4` — maximum depth of each tree (deeper = more complex but can overfit)
- `subsample=0.8` — randomly use 80% of training data per tree (reduces overfitting)
- `min_samples_leaf=5` — minimum samples per leaf (prevents very small splits)

> ⏱️ This takes about **3–6 minutes** on Colab CPU. Be patient!


In [ ]:
# ── Train Gradient Boosting ───────────────────────────────────────────────────
# Training time estimate: ~3-6 minutes on Colab CPU

print("🟠 Training Gradient Boosting Classifier...")
print(f"   Input shape : {X_gb_train.shape}")
print("   This takes ~3-6 minutes on Colab CPU...")
print()

start = time.time()

gb_model = GradientBoostingClassifier(
    n_estimators=300,       # Number of boosting rounds (trees)
    learning_rate=0.05,     # Step size — smaller = more robust but slower
    max_depth=4,            # Max tree depth — controls complexity
    subsample=0.8,          # Use 80% of data per tree — reduces overfitting
    min_samples_leaf=5,     # Min samples in a leaf — prevents tiny splits
    max_features='sqrt',    # Use sqrt(features) per split — adds randomness
    random_state=42,
    verbose=1               # Print progress every 100 iterations
)

gb_model.fit(X_gb_train, y_train)

elapsed = time.time() - start
print(f"\n✅ Gradient Boosting trained in {elapsed/60:.1f} minutes ({elapsed:.0f} seconds).")


### Step 3.3 — Validate on Validation Set


In [ ]:
# ── Evaluate on validation set ───────────────────────────────────────────────
print("🔍 Evaluating on Validation Set...")

gb_val_pred = gb_model.predict(X_gb_valid)
gb_val_prob = gb_model.predict_proba(X_gb_valid)[:, 1]

gb_val_scores = evaluate_model(
    model_name='Gradient Boosting',
    y_true=y_valid,
    y_pred=gb_val_pred,
    y_prob=gb_val_prob,
    split='Validation'
)


### Step 3.4 — Final Test Set Evaluation


In [ ]:
# ── Evaluate on test set ─────────────────────────────────────────────────────
print("🏁 Evaluating on Test Set (final honest score)...")

gb_test_pred = gb_model.predict(X_gb_test)
gb_test_prob = gb_model.predict_proba(X_gb_test)[:, 1]

gb_test_scores = evaluate_model(
    model_name='Gradient Boosting',
    y_true=y_test,
    y_pred=gb_test_pred,
    y_prob=gb_test_prob,
    split='Test'
)


### Step 3.5 — Feature Importance

Gradient Boosting tells us which hand-crafted features it found most useful.  
This is a powerful insight into **what signals really matter** for fake news detection.


In [ ]:
# ── Plot top 25 most important features ───────────────────────────────────────
importances = gb_model.feature_importances_
feat_imp = sorted(zip(HANDCRAFTED_COLS, importances), key=lambda x: x[1], reverse=True)[:25]
feat_names, feat_vals = zip(*feat_imp)

fig, ax = plt.subplots(figsize=(11, 9))
colors_imp = ['#ef6c00' if v > np.median(feat_vals) else '#ffcc80' for v in feat_vals]
bars = ax.barh(list(reversed(feat_names)), list(reversed(feat_vals)),
               color=list(reversed(colors_imp)), edgecolor='white')

ax.set_xlabel('Feature Importance (Gini Impurity Reduction)')
ax.set_title('🟠 Gradient Boosting — Top 25 Most Important Features\n(Hand-Crafted Signals)', fontweight='bold')
ax.axvline(x=np.mean(feat_vals), color='red', linestyle='--', lw=1.5, alpha=0.7, label=f'Mean importance')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("\n💡 Features at the top are the most powerful discriminators between FAKE and REAL.")
print("   Credit history features typically dominate — speaker credibility is a strong signal.")


### Step 3.6 — Save the Model


In [ ]:
# ── Save Gradient Boosting model ──────────────────────────────────────────────
gb_save_path = f'{MODEL_DIR}/gradient_boosting/gb_model.pkl'
joblib.dump(gb_model, gb_save_path)

# Save the list of feature columns used (needed for inference)
with open(f'{MODEL_DIR}/gradient_boosting/feature_cols.json', 'w') as f:
    json.dump(HANDCRAFTED_COLS, f)

with open(f'{MODEL_DIR}/gradient_boosting/scores.json', 'w') as f:
    json.dump({'validation': gb_val_scores, 'test': gb_test_scores}, f, indent=2)

print(f"✅ Gradient Boosting model saved to: {gb_save_path}")
print(f"   Val  Accuracy : {gb_val_scores['accuracy']:.4f}")
print(f"   Test Accuracy : {gb_test_scores['accuracy']:.4f}")


---
## 🟣 Part 4 — BERT (DistilBERT Fine-Tuning)

### What is BERT?
**BERT** (Bidirectional Encoder Representations from Transformers) is a deep neural network  
pre-trained on billions of words. It understands language **in context**, not just word frequency.

We use **DistilBERT** — a smaller, faster version of BERT that retains ~97% of BERT's accuracy  
but trains 60% faster and uses 40% less memory. Perfect for Colab!

**Why BERT is different:**
- Understands word meaning in context (e.g., "false claim" vs "claim is false")
- Captures sentence-level semantics, not just word counts
- Pre-trained knowledge means it already understands news/political language
- Usually outperforms TF-IDF + classical ML for text classification

**Input:** Raw cleaned statement text (`statement_bert` column from the CSV)

> ⚠️ **GPU required** for reasonable training time.  
> Enable GPU: Runtime → Change runtime type → Hardware accelerator → GPU (T4)

---


### Step 4.1 — Build PyTorch Dataset

PyTorch uses a `Dataset` class to handle data loading.  
We tokenise our statements using DistilBERT's tokeniser — converting words to token IDs.


In [ ]:
# ── Build PyTorch Dataset for BERT ────────────────────────────────────────────
# The tokeniser converts text to token IDs that DistilBERT understands

MODEL_NAME = 'distilbert-base-uncased'  # Smaller, faster BERT — good for Colab

print(f"⏳ Loading tokeniser: {MODEL_NAME} ...")
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
print("✅ Tokeniser loaded.")

# ── Custom Dataset class ──────────────────────────────────────────────────────
class LIARDataset(Dataset):
    """
    PyTorch Dataset for the LIAR statements.
    
    Each item returns:
    - input_ids      : token IDs for the statement
    - attention_mask : 1 for real tokens, 0 for padding
    - labels         : 0 (FAKE) or 1 (REAL)
    """
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,       # Cut off statements longer than max_length
            padding='max_length',  # Pad shorter statements to max_length
            max_length=max_length, # LIAR statements are short — 128 is enough
            return_tensors='pt'    # Return PyTorch tensors
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        # Returns total number of samples
        return len(self.labels)

    def __getitem__(self, idx):
        # Returns one sample as a dict of tensors
        return {
            'input_ids'      : self.encodings['input_ids'][idx],
            'attention_mask' : self.encodings['attention_mask'][idx],
            'labels'         : self.labels[idx]
        }


# ── Create datasets ────────────────────────────────────────────────────────────
print("\n⏳ Tokenising all statements (this takes ~1-2 minutes)...")

MAX_LEN    = 128   # Max token length — statements in LIAR are short (~20 words avg)
BATCH_SIZE = 32    # Samples per training step — reduce to 16 if you get OOM errors

bert_train_dataset = LIARDataset(train_df['statement_bert'], y_train, tokenizer, MAX_LEN)
bert_valid_dataset = LIARDataset(valid_df['statement_bert'], y_valid, tokenizer, MAX_LEN)
bert_test_dataset  = LIARDataset(test_df['statement_bert'],  y_test,  tokenizer, MAX_LEN)

# DataLoaders batch the data for training
train_loader = DataLoader(bert_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(bert_valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(bert_test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"✅ Datasets tokenised.")
print(f"   Train batches : {len(train_loader)}  ({len(bert_train_dataset)} samples)")
print(f"   Valid batches : {len(valid_loader)}  ({len(bert_valid_dataset)} samples)")
print(f"   Test  batches : {len(test_loader)}  ({len(bert_test_dataset)} samples)")
print(f"   Max token length : {MAX_LEN}")
print(f"   Batch size       : {BATCH_SIZE}")


### Step 4.2 — Load Pre-Trained DistilBERT

We load the pre-trained model and add a **classification head** (a small output layer)  
that maps the 768-dimensional BERT output to 2 classes (FAKE / REAL).


In [ ]:
# ── Load DistilBERT with classification head ──────────────────────────────────
print(f"⏳ Loading pre-trained model: {MODEL_NAME} ...")

bert_model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # Binary: FAKE (0) or REAL (1)
)
bert_model.to(device)  # Move to GPU if available

# Count trainable parameters
total_params = sum(p.numel() for p in bert_model.parameters())
trainable_params = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)

print(f"\n✅ DistilBERT loaded and moved to {device}.")
print(f"   Total parameters     : {total_params:,}")
print(f"   Trainable parameters : {trainable_params:,}")
print()
print("   Architecture:")
print("   DistilBERT (6 transformer layers, 768 hidden size)")
print("   → Pooling (CLS token) → Dropout → Linear(768 → 2) → Softmax")


### Step 4.3 — Configure Optimiser & Scheduler

- **AdamW** — the standard optimiser for transformers (Adam with weight decay fix)
- **Linear warm-up scheduler** — gently increases learning rate at the start,  
  then linearly decreases it. This prevents "catastrophic forgetting" of pre-trained weights.


In [ ]:
# ── Configure training hyperparameters ────────────────────────────────────────
EPOCHS       = 3      # 3 epochs is standard for BERT fine-tuning
LR           = 2e-5   # Small learning rate — we don't want to destroy pre-trained weights
WARMUP_STEPS = 100    # Warm-up: gradually increase LR for first 100 steps

# AdamW optimiser — correct weight decay for transformers
optimizer = AdamW(bert_model.parameters(), lr=LR, weight_decay=0.01)

# Total training steps = batches per epoch × number of epochs
total_steps = len(train_loader) * EPOCHS

# Linear warm-up then linear decay schedule
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)

print(f"✅ Optimiser and scheduler configured.")
print(f"   Learning rate        : {LR}")
print(f"   Epochs               : {EPOCHS}")
print(f"   Total training steps : {total_steps}")
print(f"   Warm-up steps        : {WARMUP_STEPS}")
print(f"   Weight decay         : 0.01")


### Step 4.4 — Training Loop

We train for 3 epochs. Each epoch:
1. **Forward pass** — feed batch through the model, compute loss
2. **Backward pass** — compute gradients (how to update each parameter)
3. **Optimiser step** — update model weights
4. **Validation** — check performance on the validation set after each epoch

> ⏱️ With GPU (T4): ~5-7 minutes total.  
> Without GPU: ~60-90 minutes (not recommended).


In [ ]:
# ── BERT Training Loop ────────────────────────────────────────────────────────
# Each epoch: train on all batches → validate → print progress

def bert_evaluate(model, loader, device):
    """Run model on a DataLoader, return (predictions, probabilities, true labels)."""
    model.eval()   # Switch to evaluation mode (disables dropout)
    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():  # Don't compute gradients during evaluation
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits  = outputs.logits

            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_preds), np.array(all_probs), np.array(all_labels)


# ── Run training ───────────────────────────────────────────────────────────────
print("🟣 Starting BERT Fine-Tuning...")
print(f"   Device : {device}")
print(f"   Epochs : {EPOCHS}")
print()

history = {'train_loss': [], 'val_acc': [], 'val_f1': []}
best_val_acc = 0.0
best_epoch   = 0

for epoch in range(1, EPOCHS + 1):
    # ── Training phase ───────────────────────────────────────────────────────
    bert_model.train()   # Switch to training mode (enables dropout)
    epoch_loss   = 0
    n_batches    = len(train_loader)
    epoch_start  = time.time()

    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()  # Clear gradients from previous step

        # Forward pass — compute loss
        outputs = bert_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        epoch_loss += loss.item()

        # Backward pass — compute gradients
        loss.backward()

        # Clip gradients to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), max_norm=1.0)

        optimizer.step()   # Update weights
        scheduler.step()   # Update learning rate

        # Print progress every 50 steps
        if (step + 1) % 50 == 0:
            avg_loss = epoch_loss / (step + 1)
            print(f"   Epoch {epoch}/{EPOCHS} | Step {step+1}/{n_batches} | Loss: {avg_loss:.4f}")

    avg_epoch_loss = epoch_loss / n_batches

    # ── Validation phase ─────────────────────────────────────────────────────
    val_preds, val_probs, val_labels = bert_evaluate(bert_model, valid_loader, device)
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1  = f1_score(val_labels, val_preds)

    history['train_loss'].append(avg_epoch_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    elapsed = time.time() - epoch_start
    print(f"\n📈 Epoch {epoch}/{EPOCHS} complete in {elapsed:.0f}s")
    print(f"   Train Loss    : {avg_epoch_loss:.4f}")
    print(f"   Val Accuracy  : {val_acc:.4f}  ({val_acc*100:.2f}%)")
    print(f"   Val F1 Score  : {val_f1:.4f}")

    # Save best model based on validation accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch   = epoch
        bert_model.save_pretrained(f'{MODEL_DIR}/bert/best_model')
        tokenizer.save_pretrained(f'{MODEL_DIR}/bert/best_model')
        print(f"   💾 Best model saved! (Val Acc: {best_val_acc:.4f})")
    print()

print(f"\n✅ BERT training complete!")
print(f"   Best validation accuracy: {best_val_acc:.4f} at epoch {best_epoch}")


### Step 4.5 — Plot Training History

Visualise how the model improved over each epoch.


In [ ]:
# ── Plot training history ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training Loss
axes[0].plot(range(1, EPOCHS+1), history['train_loss'],
             'o-', color='#7b1fa2', lw=2, markersize=8, label='Train Loss')
axes[0].set_title('Training Loss per Epoch', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_xticks(range(1, EPOCHS+1))
axes[0].grid(True, alpha=0.4)
axes[0].legend()

# Validation Accuracy & F1
axes[1].plot(range(1, EPOCHS+1), [v*100 for v in history['val_acc']],
             'o-', color='#1976d2', lw=2, markersize=8, label='Val Accuracy %')
axes[1].plot(range(1, EPOCHS+1), [v*100 for v in history['val_f1']],
             's--', color='#388e3c', lw=2, markersize=8, label='Val F1 %')
axes[1].set_title('Validation Performance per Epoch', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score (%)')
axes[1].set_xticks(range(1, EPOCHS+1))
axes[1].grid(True, alpha=0.4)
axes[1].legend()

plt.suptitle('🟣 BERT Fine-Tuning — Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### Step 4.6 — Final Test Set Evaluation

Load the best saved model and evaluate it on the test set.


In [ ]:
# ── Load best model and evaluate on test set ──────────────────────────────────
print("⏳ Loading best saved model...")

best_bert = DistilBertForSequenceClassification.from_pretrained(
    f'{MODEL_DIR}/bert/best_model'
)
best_bert.to(device)
print("✅ Best model loaded.")

# ── Validation set (final) ─────────────────────────────────────────────────────
print("\n🔍 Evaluating on Validation Set...")
bert_val_preds, bert_val_probs, bert_val_labels = bert_evaluate(best_bert, valid_loader, device)

bert_val_scores = evaluate_model(
    model_name='BERT (DistilBERT)',
    y_true=bert_val_labels,
    y_pred=bert_val_preds,
    y_prob=bert_val_probs,
    split='Validation'
)

# ── Test set (final honest score) ─────────────────────────────────────────────
print("\n🏁 Evaluating on Test Set (final honest score)...")
bert_test_preds, bert_test_probs, bert_test_labels = bert_evaluate(best_bert, test_loader, device)

bert_test_scores = evaluate_model(
    model_name='BERT (DistilBERT)',
    y_true=bert_test_labels,
    y_pred=bert_test_preds,
    y_prob=bert_test_probs,
    split='Test'
)

# Save scores
with open(f'{MODEL_DIR}/bert/scores.json', 'w') as f:
    json.dump({'validation': bert_val_scores, 'test': bert_test_scores}, f, indent=2)

print(f"\n✅ BERT evaluation complete!")
print(f"   Val  Accuracy : {bert_val_scores['accuracy']:.4f}")
print(f"   Test Accuracy : {bert_test_scores['accuracy']:.4f}")


---
## 📊 Part 5 — Cross-Model Comparison

Now we compare all three models side-by-side.

This is the **final summary** — the most important cell in the notebook!


In [ ]:
# ── Build comparison table ────────────────────────────────────────────────────
# Collect all scores into a single DataFrame for easy comparison

results = pd.DataFrame([
    {
        'Model'         : 'Logistic Regression',
        'Features'      : 'TF-IDF + Hand-crafted',
        'Val Accuracy'  : lr_val_scores['accuracy'],
        'Test Accuracy' : lr_test_scores['accuracy'],
        'Test F1'       : lr_test_scores['f1'],
        'Test Precision': lr_test_scores['precision'],
        'Test Recall'   : lr_test_scores['recall'],
    },
    {
        'Model'         : 'Gradient Boosting',
        'Features'      : 'Hand-crafted only',
        'Val Accuracy'  : gb_val_scores['accuracy'],
        'Test Accuracy' : gb_test_scores['accuracy'],
        'Test F1'       : gb_test_scores['f1'],
        'Test Precision': gb_test_scores['precision'],
        'Test Recall'   : gb_test_scores['recall'],
    },
    {
        'Model'         : 'BERT (DistilBERT)',
        'Features'      : 'Raw text (fine-tuned)',
        'Val Accuracy'  : bert_val_scores['accuracy'],
        'Test Accuracy' : bert_test_scores['accuracy'],
        'Test F1'       : bert_test_scores['f1'],
        'Test Precision': bert_test_scores['precision'],
        'Test Recall'   : bert_test_scores['recall'],
    }
])

# Format percentages
for col in ['Val Accuracy','Test Accuracy','Test F1','Test Precision','Test Recall']:
    results[col] = results[col].apply(lambda x: f'{x*100:.2f}%')

print("\n🏆 FINAL MODEL COMPARISON")
print("=" * 70)
print(results.to_string(index=False))
print("=" * 70)


In [ ]:
# ── Visualise comparison ─────────────────────────────────────────────────────
models     = ['Logistic\nRegression', 'Gradient\nBoosting', 'BERT\n(DistilBERT)']
test_accs  = [lr_test_scores['accuracy'],  gb_test_scores['accuracy'],  bert_test_scores['accuracy']]
test_f1s   = [lr_test_scores['f1'],        gb_test_scores['f1'],        bert_test_scores['f1']]
val_accs   = [lr_val_scores['accuracy'],   gb_val_scores['accuracy'],   bert_val_scores['accuracy']]

colors_model = ['#1976d2', '#e65100', '#7b1fa2']

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# ── Test Accuracy ─────────────────────────────────────────────────────────────
bars = axes[0].bar(models, [a*100 for a in test_accs], color=colors_model, edgecolor='white', width=0.5)
axes[0].bar_label(bars, labels=[f'{a*100:.2f}%' for a in test_accs], padding=4, fontsize=11, fontweight='bold')
axes[0].set_title('Test Set Accuracy', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_ylim(0, 100)
axes[0].axhline(50, color='grey', lw=1, linestyle='--', alpha=0.5, label='Random baseline')
axes[0].legend(fontsize=9)

# ── Test F1 Score ─────────────────────────────────────────────────────────────
bars2 = axes[1].bar(models, [f*100 for f in test_f1s], color=colors_model, edgecolor='white', width=0.5)
axes[1].bar_label(bars2, labels=[f'{f*100:.2f}%' for f in test_f1s], padding=4, fontsize=11, fontweight='bold')
axes[1].set_title('Test Set F1 Score', fontweight='bold', fontsize=12)
axes[1].set_ylabel('F1 Score (%)')
axes[1].set_ylim(0, 100)

# ── Val vs Test Accuracy ──────────────────────────────────────────────────────
x = np.arange(len(models))
width = 0.35
b1 = axes[2].bar(x - width/2, [v*100 for v in val_accs],  width, label='Validation', color=[c+'aa' for c in ['#1976d2','#e65100','#7b1fa2']], edgecolor='white')
b2 = axes[2].bar(x + width/2, [v*100 for v in test_accs], width, label='Test',       color=colors_model, edgecolor='white')
axes[2].set_xticks(x)
axes[2].set_xticklabels(models)
axes[2].set_title('Validation vs Test Accuracy', fontweight='bold', fontsize=12)
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_ylim(0, 100)
axes[2].legend(fontsize=10)
axes[2].bar_label(b1, labels=[f'{v*100:.1f}%' for v in val_accs],  padding=3, fontsize=9)
axes[2].bar_label(b2, labels=[f'{v*100:.1f}%' for v in test_accs], padding=3, fontsize=9)

plt.suptitle('🏆 Model Comparison — Fake News Detection on LIAR Dataset',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Key Takeaways:")
best_model = models[np.argmax(test_accs)].replace('\n', ' ')
print(f"   🥇 Best model by test accuracy : {best_model} ({max(test_accs)*100:.2f}%)")
print(f"   🔵 Logistic Regression is fastest to train and most interpretable")
print(f"   🟠 Gradient Boosting captures non-linear patterns in hand-crafted features")
print(f"   🟣 BERT understands language context — highest ceiling, most compute")


---
## ✅ Final Summary

### What We Built
| Model | Approach | Best For |
|-------|----------|---------|
| **Logistic Regression** | Linear classifier on TF-IDF | Baseline, interpretability, speed |
| **Gradient Boosting** | Ensemble trees on engineered features | Credibility/linguistic signals |
| **BERT (DistilBERT)** | Transformer fine-tuning on raw text | Best accuracy, deep language understanding |

### Cross-Domain Generalisation
The LIAR dataset is from **political speech (PolitiFact)**.  
Models may not generalise to other domains (social media, news articles) without re-training.  
To test cross-domain performance, evaluate on a different dataset (e.g., FakeNewsNet) using the saved models.

### Saved Files
All models are saved in your Google Drive under `LIAR_Models/`:
- `logistic_regression/lr_model.pkl`
- `gradient_boosting/gb_model.pkl`
- `bert/best_model/` (HuggingFace format)

### Next Steps
- Try **hyperparameter tuning** with `GridSearchCV` for Logistic Regression / Gradient Boosting
- Try **BERT-large** or `roberta-base` for potentially better accuracy
- Evaluate **cross-domain generalisation** on FakeNewsNet dataset
- Build a simple **inference pipeline** to predict new unseen statements

---
*Dataset: LIAR — William Yang Wang, ACL 2017 | Project #13: Fake News Detection Using NLP Classifiers*
